# 技術規格書 RAG 前導實驗（Colab）

**執行前：** 上方選單「執行階段」→「變更執行階段類型」→ 選 **T4 GPU** → 儲存。分配不到 GPU 也能跑，但要跳過 7B 模型。

每一格左邊的 ▶ 按下去就會執行。一格跑完（出現綠色勾勾）再按下一格。文字格不用執行。

## 1. 下載程式

In [ ]:
!git clone https://github.com/chunyang130/datasheet-rag-pilot.git
%cd datasheet-rag-pilot
!pip install -q -r requirements.txt
!ls

## 2. 讀取 API 金鑰

先在左側 🔑「密鑰」新增 `ANTHROPIC_API_KEY`，並打開「Notebook 存取權」。

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print('ANTHROPIC_API_KEY 已載入')

## 3. 上傳 3 份規格書 PDF

執行後會出現「選擇檔案」按鈕，一次選取 3 份 PDF。檔名不用改，程式會自動辨認。

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('data', exist_ok=True)
for name in files.upload():
    n = name.lower()
    target = 'lm317.pdf' if 'lm317' in n else 'wroom.pdf' if 'wroom' in n else 'esp32.pdf'
    shutil.move(name, f'data/{target}')
    print(name, '→', f'data/{target}')
!ls data

## 4. 切段與自動測試

成功時會顯示「共 282 段」與「OK」。

In [ ]:
!python ingest.py
!python -m unittest test_rag.py

## 5. 安裝 Ollama 與下載模型（約 3～8 分鐘）

出現紅字的 WARNING 是正常的。**沒有 GPU 時，先刪掉 `qwen2.5:7b` 那一行再執行。**

In [ ]:
!apt-get -qq install -y zstd   # 新版 Ollama 安裝檔需要 zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama pull bge-m3
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b
!ollama list

## 6. 檢索評估（不花錢）

每一行都會印出「命中 x/27」。

In [ ]:
!python eval_retrieval.py --mode bm25 --out results/retrieval_bm25.json | head -6
!python eval_retrieval.py --mode bm25 --glossary --out results/retrieval_bm25_gl.json | head -6
!python eval_retrieval.py --mode vector --out results/retrieval_vector.json | head -6
!python eval_retrieval.py --mode hybrid --glossary --out results/retrieval_hybrid_gl.json | head -6

## 7. 雲端模型：Claude Haiku（驗證開、關各一次）

In [ ]:
!python rag.py run --provider claude --model claude-haiku-5-5 --retriever hybrid --glossary
!python rag.py run --provider claude --model claude-haiku-5-5 --retriever hybrid --glossary --no-verify

## 8. 地端模型：Qwen2.5

**沒有 GPU 時，把 `'qwen2.5:7b'` 從清單刪掉**，改成 `['qwen2.5:3b']`。

In [ ]:
for m in ['qwen2.5:3b', 'qwen2.5:7b']:
    for v in ['--verify', '--no-verify']:
        !python rag.py run --provider ollama --model {m} --retriever hybrid --glossary {v}

## 9. 比較表與下載結果

會自動下載 `results.zip`，請把它傳回給 Claude。

In [ ]:
!python rag.py report results/claude*.json results/ollama*.json
!zip -qr results.zip results
files.download('results.zip')